In [1]:
# installing libraries if needed
%pip install pyiqa
%pip install pandas
%pip install scikit-learn
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# importing libraries
import os
import random

import numpy as np
import pandas as pd

import torch
import pyiqa

from tqdm import tqdm

from sklearn.metrics.pairwise import cosine_similarity

In [11]:
# paths
ROOT_DIR = "natural_dataset"

GALLERY_DIR = os.path.join(
    ROOT_DIR,
    "gallery",
)

PROBE_DIR = os.path.join(
    ROOT_DIR,
    "probe",
)

GALLERY_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "gallery_embeddings",
)

PROBE_EMBED_DIR = os.path.join(
    ROOT_DIR,
    "probe_embeddings",
)

OUTPUT_DATASET = "tinyface_like_dataset.csv"

OLD_DATASET = "train_mlp_extended_all.csv"

FINAL_DATASET = "train_mlp_extended_final_version.csv"

In [12]:
# loading MUSIQ for quality score check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

musiq_metric = pyiqa.create_metric(
    "musiq",
    device=device,
)

print("MUSIQ Loaded")

Device : cpu
Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth
MUSIQ Loaded


In [13]:
# quality score helper functions
# -------------------------
# MUSIQ Quality
# -------------------------


def get_quality_score(image_path):

    score = musiq_metric(image_path)

    return float(score.cpu().item())


# -------------------------
# Gallery Loader
# -------------------------


def load_gallery_database(gallery_root):

    gallery_db = {}

    total = 0

    for identity in sorted(os.listdir(gallery_root)):

        identity_dir = os.path.join(
            gallery_root,
            identity,
        )

        if not os.path.isdir(identity_dir):
            continue

        gallery_db[identity] = []

        for emb_file in sorted(os.listdir(identity_dir)):

            if not emb_file.endswith(".npy"):
                continue

            emb = np.load(
                os.path.join(
                    identity_dir,
                    emb_file,
                )
            )

            gallery_db[identity].append(emb)

            total += 1

    print("Gallery Identities :", len(gallery_db))
    print("Gallery Embeddings :", total)

    return gallery_db


# -------------------------
# Probe Loader
# -------------------------


def load_probe_database(probe_root):

    probe_db = {}

    total = 0

    for identity in sorted(os.listdir(probe_root)):

        identity_dir = os.path.join(
            probe_root,
            identity,
        )

        if not os.path.isdir(identity_dir):
            continue

        files = [f for f in os.listdir(identity_dir) if f.endswith(".npy")]

        if len(files) == 0:
            continue

        emb = np.load(
            os.path.join(
                identity_dir,
                files[0],
            )
        )

        probe_db[identity] = emb

        total += 1

    print("Probe Embeddings :", total)

    return probe_db

In [14]:
# loading gallery and probe embeddings
gallery_db = load_gallery_database(GALLERY_EMBED_DIR)

probe_db = load_probe_database(PROBE_EMBED_DIR)

FileNotFoundError: [Errno 2] No such file or directory: 'natural_dataset/gallery_embeddings'

In [7]:
# gallery search
def search_gallery(
    probe_embedding,
    gallery_db,
):

    identity_scores = {}

    for identity in gallery_db:

        sims = []

        for gallery_embedding in gallery_db[identity]:

            sim = cosine_similarity(
                probe_embedding.reshape(1, -1),
                gallery_embedding.reshape(1, -1),
            )[0][0]

            sims.append(sim)

        identity_scores[identity] = max(sims)

    sorted_scores = sorted(
        identity_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    return (
        identity_scores,
        sorted_scores,
    )

In [8]:
# dataset generation
rows = []

counter = 0

for identity in tqdm(probe_db):

    probe_embedding = probe_db[identity]

    probe_folder = os.path.join(
        PROBE_DIR,
        identity,
    )

    image_name = None

    for file in os.listdir(probe_folder):

        if file.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
            )
        ):
            image_name = file
            break

    if image_name is None:
        continue

    image_path = os.path.join(
        probe_folder,
        image_name,
    )

    quality_score = get_quality_score(
        image_path,
    )

    (
        identity_scores,
        sorted_scores,
    ) = search_gallery(
        probe_embedding,
        gallery_db,
    )

    ##################################################
    # Genuine Pair
    ##################################################

    genuine_similarity = identity_scores[identity]

    best_impostor_similarity = max(
        score for gallery_id, score in identity_scores.items() if gallery_id != identity
    )

    genuine_margin = genuine_similarity - best_impostor_similarity

    rows.append(
        {
            "quality_score": quality_score,
            "best_similarity": genuine_similarity,
            "margin": genuine_margin,
            "label": 1,
            "true_identity": identity,
            "gallery_identity": identity,
        }
    )

    ##################################################
    # Hard Negatives
    ##################################################

    impostor_scores = [
        (gallery_id, similarity)
        for gallery_id, similarity in sorted_scores
        if gallery_id != identity
    ]

    hard_negative_identity = impostor_scores[0][0]
    hard_negative_similarity = impostor_scores[0][1]
    second_impostor_similarity = impostor_scores[1][1]

    hard_negative_margin = hard_negative_similarity - second_impostor_similarity
    rows.append(
        {
            "quality_score": quality_score,
            "best_similarity": hard_negative_similarity,
            "margin": hard_negative_margin,
            "label": 0,
            "true_identity": identity,
            "gallery_identity": hard_negative_identity,
        }
    )

  0%|          | 0/200 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'naturally_dataset/probe/Al_Gore'

In [9]:
print("Embedding exists:",
      os.path.exists("naturally_dataset/probe_embeddings/Al_Gore"))

print("Image exists:",
      os.path.exists("naturally_dataset/probe/Al_Gore"))

Embedding exists: True
Image exists: False


In [10]:
print("Embedding identities:", len(os.listdir(PROBE_EMBED_DIR)))
print("Image identities:", len(os.listdir(PROBE_DIR)))

Embedding identities: 200


FileNotFoundError: [Errno 2] No such file or directory: 'naturally_dataset/probe'

In [ ]:
# saving the dataset
dataset = pd.DataFrame(rows)

dataset.to_csv(
    OUTPUT_DATASET,
    index=False,
)

print()

print("Dataset Saved")

print()

print(dataset.shape)

print()

print(dataset.head())

In [ ]:
# dataset stats for verification
print()

print("Label Counts")

print(dataset["label"].value_counts())

print()

print("Quality Statistics")

print(dataset["quality_score"].describe())

print()

print("Similarity Statistics")

print(dataset["best_similarity"].describe())

print()

print("Margin Statistics")

print(dataset["margin"].describe())

In [ ]:
# appending the new dataset to existing training dataset
old_dataset = pd.read_csv(
    OLD_DATASET,
)

print(
    "Old Dataset:",
    old_dataset.shape,
)

print(
    "New Dataset:",
    dataset.shape,
)

combined_dataset = pd.concat(
    [
        old_dataset,
        dataset,
    ],
    ignore_index=True,
)

combined_dataset = combined_dataset.sample(
    frac=1,
    random_state=42,
).reset_index(
    drop=True,
)

combined_dataset.to_csv(
    FINAL_DATASET,
    index=False,
)

print()

print("Combined Dataset Saved")

print(
    combined_dataset.shape,
)

In [ ]:
# final stats
print()

print("Final Dataset Statistics")

print()

print(combined_dataset["label"].value_counts())

print()

print(combined_dataset.describe())